# Unidad I — Paso 0: Verificación del entorno

**Proyecto Sello:** CitiBike Intelligent Mobility — Data Geniuses
**Dimensión U1 (batch):** Utilización histórica de estaciones
**Integrante:** Grimaldo Arredondo Martinez

Objetivo de este notebook: dejar evidencia de que el entorno distribuido
levanta correctamente y de que el **esquema explícito** es una necesidad
técnica de estos datos, no un requisito formal.

Este notebook no produce la salida Gold ni el modelo; sólo verifica el entorno.


## 1. Evidencia de ejecución

Fecha, usuario y host de la corrida (criterio 6 de la rúbrica).

> La captura de pantalla de esta celda debe mostrar **también el reloj del sistema operativo**, no sólo esta salida.

In [ ]:
import getpass, socket, sys, platform
from datetime import datetime

print("Fecha y hora :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Usuario      :", getpass.getuser())
print("Host         :", socket.gethostname())
print("Python       :", sys.version.split()[0])
print("Sistema      :", platform.platform())

## 2. Sesión de Spark

Configuración relevante:

- `local[*]` — Spark usa todos los núcleos disponibles del contenedor. Es modo local, pero el modelo de ejecución (DAG, tareas, particiones, shuffle) es el mismo de un clúster real.
- `spark.sql.shuffle.partitions = 8` — el valor por defecto es **200**, pensado para clúster. En local eso genera 200 tareas diminutas por cada `groupBy` y hace todo lentísimo.
- `spark.driver.memory = 4g` — en modo local el driver hace todo el trabajo. Debe ser menor que el `mem_limit` del contenedor.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CitiBike-U1-Estaciones-Verificacion")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("Núcleos       :", spark.sparkContext.defaultParallelism)
print("Spark UI      : http://localhost:4040")

## 3. Localización de los datos

Los CSV deben estar en `data/raw/` de tu repositorio. Dentro del contenedor esa carpeta se ve como `/home/jovyan/work/data/raw`.

In [ ]:
import os, glob

RUTA_RAW = "/home/jovyan/work/data/raw"

archivos = sorted(glob.glob(f"{RUTA_RAW}/*.csv"))
if not archivos:
    raise FileNotFoundError(f"No hay CSV en {RUTA_RAW}. Copia ahí los archivos de CitiBike.")

total_mb = 0
for a in archivos:
    mb = os.path.getsize(a) / 1024**2
    total_mb += mb
    print(f"  {os.path.basename(a):45s} {mb:8.1f} MB")
print(f"\n{len(archivos)} archivo(s), {total_mb:.1f} MB en total")

## 4. Por qué el esquema explícito no es opcional

`start_station_id` tiene valores como `6364.10` y `6107.08`. **Parecen** números
decimales, pero son identificadores: el `.10` final no es una décima, es parte
del código de la estación.

Si Spark infiere el tipo, lo lee como `double` y `6364.10` se convierte en
`6364.1`. Las agrupaciones por estación se corrompen **sin lanzar ningún error**.

La celda siguiente demuestra el problema antes de resolverlo.

In [ ]:
from pyspark.sql.functions import col

# Lectura INGENUA: dejando que Spark adivine los tipos
df_malo = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(archivos[0])
    .limit(200000)
)

print("Tipo inferido para start_station_id:",
      dict(df_malo.dtypes)["start_station_id"])

print("\nValores tal como quedan:")
df_malo.select("start_station_name", "start_station_id").distinct().show(5, truncate=False)

Ahora la lectura correcta, con `StructType` declarado columna por columna.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType
)

esquema_viajes = StructType([
    StructField("ride_id",            StringType(),    True),
    StructField("rideable_type",      StringType(),    True),
    StructField("started_at",         TimestampType(), True),
    StructField("ended_at",           TimestampType(), True),
    StructField("start_station_name", StringType(),    True),
    StructField("start_station_id",   StringType(),    True),   # <- identificador, NO número
    StructField("end_station_name",   StringType(),    True),
    StructField("end_station_id",     StringType(),    True),   # <- identificador, NO número
    StructField("start_lat",          DoubleType(),    True),
    StructField("start_lng",          DoubleType(),    True),
    StructField("end_lat",            DoubleType(),    True),
    StructField("end_lng",            DoubleType(),    True),
    StructField("member_casual",      StringType(),    True),
])

df = (
    spark.read
    .option("header", True)
    # Los timestamps traen milisegundos: 2026-07-04 15:52:29.356
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss.SSS")
    .schema(esquema_viajes)
    .csv(f"{RUTA_RAW}/*.csv")
)

print("Tipo declarado para start_station_id:",
      dict(df.dtypes)["start_station_id"])

df.select("start_station_name", "start_station_id").distinct().show(5, truncate=False)

## 5. Verificación de carga

Tres comprobaciones: que el conteo cuadre, que ningún timestamp haya fallado al parsear (un formato mal declarado produce `null` silenciosos) y que los valores se vean sanos.

In [ ]:
from pyspark.sql.functions import col, min as _min, max as _max

n = df.count()
print(f"Filas leídas: {n:,}")

nulos_ts = df.filter(col("started_at").isNull() | col("ended_at").isNull()).count()
print(f"Timestamps no parseados: {nulos_ts}")
if nulos_ts > 0:
    print("  ¡ATENCIÓN! Revisa la opción timestampFormat.")

df.select(_min("started_at").alias("desde"),
          _max("started_at").alias("hasta")).show(truncate=False)

df.show(3, truncate=False)

## 6. Evidencia de ejecución perezosa

Spark no ejecutó nada en las celdas anteriores hasta que apareció una **acción** (`count()`, `show()`). Las transformaciones sólo construyen el plan.

Esta celda es la base del requisito 2.6 de S2; en el notebook 01 se desarrolla completo.

In [ ]:
from pyspark.sql.functions import col

# TRANSFORMACIÓN: no dispara cómputo, sólo agrega un nodo al plan
df_filtrado = df.filter(col("start_station_id").isNotNull())
print("Transformación declarada. ¿Spark leyó algo? No todavía.\n")

# El plan existe aunque el cómputo no haya ocurrido
df_filtrado.explain(mode="simple")

# ACCIÓN: aquí sí se ejecuta el DAG
print("\nFilas tras el filtro:", df_filtrado.count())

## 7. Checklist del Paso 0

Si las 6 secciones anteriores corrieron sin error:

- [ ] Jupyter Lab accesible en `localhost:8888`
- [ ] Spark UI accesible en `localhost:4040` mientras la sesión está viva
- [ ] Los CSV se leen desde `data/raw/`
- [ ] El esquema explícito conserva los IDs de estación como texto
- [ ] Los timestamps parsean sin `null`
- [ ] Captura guardada en `evidencias/` con reloj del sistema y usuario visibles

**Siguiente:** notebook `01_calidad_y_gold.ipynb` — controles de calidad,
transformaciones, unión de salidas y llegadas por estación, y la salida Gold
particionada.

In [ ]:
# Deja la sesión viva mientras trabajas.
# Ejecuta esto sólo al terminar: libera el puerto 4040 y la memoria.
# spark.stop()